# 实验九 主成分分析

主成分分析（PCA）是一种非监督的线性降维方法，通过线性变换将高维数据映射到低维空间，同时尽可能保留原始数据中的重要信息。

本实验先使用鸢尾花数据完成 PCA 二维降维和可视化，再使用乳腺癌数据比较不同 `n_components`、标准化方式和累计方差保留比例下的分类结果。

## 1. 使用鸢尾花数据完成基础 PCA 降维

将鸢尾花数据从 4 维降至 2 维，输出主成分方向和降维后数据形状，并绘制二维散点图。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.decomposition import PCA

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

iris = datasets.load_iris()
X, y = iris.data, iris.target

pca = PCA(n_components=2)
X_reduction = pca.fit_transform(X)
print('主成分方向：\n', pca.components_)
print('降维后形状：', X_reduction.shape)

plt.scatter(X_reduction[:, 0], X_reduction[:, 1], c=y)
plt.xlabel('第一主成分')
plt.ylabel('第二主成分')
plt.title('鸢尾花数据 PCA 降维结果')
plt.show()

> **操作要求**：记录降维前后的数据形状，观察三个类别在二维主成分空间中的分布情况。

## 2. 修改主成分数量 n_components

将 `n_components` 分别设置为 1、2、3、4，观察数据形状和累计方差解释率变化。

In [ ]:
for n in [1, 2, 3, 4]:
    model = PCA(n_components=n)
    X_new = model.fit_transform(X)
    total_ratio = np.sum(model.explained_variance_ratio_)
    print('n_components =', n, '数据形状 =', X_new.shape, '累计方差 =', round(total_ratio, 4))

> **操作要求**：比较 n_components=1、2、3、4 的结果，分析保留维度增加后信息保留比例如何变化。

## 3. 比较标准化前后的 PCA 结果

分别对原始乳腺癌数据和标准化数据进行 PCA，比较前两个主成分的方差解释率。

In [ ]:
from sklearn.preprocessing import StandardScaler

cancer = datasets.load_breast_cancer()
X, y = cancer.data, cancer.target

# 未标准化数据
pca_raw = PCA(n_components=2)
X_raw = pca_raw.fit_transform(X)
print('未标准化：', pca_raw.explained_variance_ratio_)

# 标准化数据
X_std = StandardScaler().fit_transform(X)
pca_std = PCA(n_components=2)
X_std_2d = pca_std.fit_transform(X_std)
print('标准化后：', pca_std.explained_variance_ratio_)

plt.figure()
plt.scatter(X_std_2d[:, 0], X_std_2d[:, 1], c=y)
plt.title('标准化后的 PCA 结果')
plt.show()

> **操作要求**：比较两组方差解释率，分析特征量纲差异是否会影响 PCA 主成分方向。

## 4. 修改累计方差保留比例

将 `n_components` 设置为 0.80、0.90、0.95、0.99，记录自动选择的主成分数量。

In [ ]:
for ratio in [0.80, 0.90, 0.95, 0.99]:
    model = PCA(n_components=ratio)
    X_new = model.fit_transform(X_std)
    print('保留比例 =', ratio, '主成分数 =', model.n_components_, '数据形状 =', X_new.shape)

# 绘制累计方差解释率曲线
pca_all = PCA()
pca_all.fit(X_std)
cum_ratio = np.cumsum(pca_all.explained_variance_ratio_)
plt.plot(range(1, len(cum_ratio) + 1), cum_ratio)
plt.xlabel('主成分数')
plt.ylabel('累计方差解释率')
plt.title('累计方差解释率曲线')
plt.show()

> **操作要求**：记录不同保留比例对应的主成分数量，观察保留信息越多时维度是否越高。

## 5. 比较不同 PCA 维度下的分类结果

使用不同维度的 PCA 数据训练逻辑回归，记录训练准确率和测试准确率。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

for n in [2, 5, 10, 0.95]:
    X_new = PCA(n_components=n).fit_transform(X_std)
    X_train, X_test, y_train, y_test = \
        train_test_split(X_new, y, test_size=0.25, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    print('n =', n, '训练 =', round(clf.score(X_train, y_train), 4),
          '测试 =', round(clf.score(X_test, y_test), 4))

> **操作要求**：比较不同维度下训练集和测试集准确率，分析降维过多或保留维度过多带来的影响。

## 6. 创新拓展实践：二维分类决策边界

将乳腺癌数据降至二维，训练逻辑回归模型，并尝试绘制分类决策边界。

任务：使用 `DecisionBoundaryDisplay` 绘制二维分类边界，再将 PCA 设置为 0.95，比较二维可视化结果与保留 95% 信息时的分类准确率。

> 注意：PCA 对特征尺度较敏感，乳腺癌数据进行降维前应先完成标准化。

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_std)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_2d, y)
print('二维分类准确率：', clf.score(X_2d, y))

# 绘制二维分类决策边界
plt.figure(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(
    clf, X_2d, cmap=plt.cm.Paired, response_method="predict", ax=plt.gca()
)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, s=50, edgecolors='k')
plt.title('乳腺癌数据 PCA 二维分类决策边界')
plt.show()

# 保留 95% 信息时的分类准确率
X_95 = PCA(n_components=0.95).fit_transform(X_std)
X_train95, X_test95, y_train95, y_test95 = train_test_split(
    X_95, y, test_size=0.25, random_state=42, stratify=y)
clf95 = LogisticRegression(max_iter=1000)
clf95.fit(X_train95, y_train95)
print('保留 95% 信息时的测试准确率：', round(clf95.score(X_test95, y_test95), 4))